In [4]:
from clarifai_grpc.channel.clarifai_channel import ClarifaiChannel
from clarifai_grpc.grpc.api import resources_pb2, service_pb2, service_pb2_grpc

PAT = "1a650136d25745568540fa41676ddd29"  # ← ta clé PAT

channel  = ClarifaiChannel.get_grpc_channel()
stub     = service_pb2_grpc.V2Stub(channel)
metadata = (("authorization", f"Key {PAT}"),)

response = stub.ListModels(
    service_pb2.ListModelsRequest(
        user_app_id = resources_pb2.UserAppIDSet(
            user_id = "clarifai",
            app_id  = "main"
        ),
        page     = 1,
        per_page = 20
    ),
    metadata=metadata
)

print(f"📦 Modèles disponibles :\n")
for model in response.models:
    if "food" in model.name.lower() or "food" in model.id.lower():
        print(f"   ✅ ID      : {model.id}")
        print(f"      Nom    : {model.name}")
        print(f"      Version: {model.model_version.id}")
        print()

📦 Modèles disponibles :

   ✅ ID      : food-item-v1-recognition
      Nom    : food-items-v1.0
      Version: dfebc169854e429086aceb8368662641



In [5]:
from clarifai_grpc.channel.clarifai_channel import ClarifaiChannel
from clarifai_grpc.grpc.api import resources_pb2, service_pb2, service_pb2_grpc
from clarifai_grpc.grpc.api.status import status_code_pb2
from pathlib import Path

PAT       = "1a650136d25745568540fa41676ddd29"  # ← ta clé PAT
USER_ID   = "clarifai"
APP_ID    = "main"
MODEL_ID  = "food-item-v1-recognition"
MODEL_VER = "dfebc169854e429086aceb8368662641"

def clarifai_detect_food(image_path, pat=PAT):
    channel  = ClarifaiChannel.get_grpc_channel()
    stub     = service_pb2_grpc.V2Stub(channel)
    metadata = (("authorization", f"Key {pat}"),)

    with open(image_path, "rb") as f:
        image_bytes = f.read()

    response = stub.PostModelOutputs(
        service_pb2.PostModelOutputsRequest(
            user_app_id = resources_pb2.UserAppIDSet(
                user_id = USER_ID,
                app_id  = APP_ID
            ),
            model_id   = MODEL_ID,
            version_id = MODEL_VER,
            inputs     = [
                resources_pb2.Input(
                    data=resources_pb2.Data(
                        image=resources_pb2.Image(base64=image_bytes)
                    )
                )
            ]
        ),
        metadata=metadata
    )

    if response.status.code != status_code_pb2.SUCCESS:
        print(f"❌ Erreur : {response.status.description}")
        return None

    return [
        {"name": c.name, "score": round(c.value*100, 1)}
        for c in response.outputs[0].data.concepts[:10]
    ]

# ── Test sur 4 plats ──────────────────────────────────────
images_dir = Path("C:/SmartMeal/data/food101/food-101/food-101/images")

test_foods = {
    "pizza"          : list((images_dir / "pizza").glob("*.jpg"))[0],
    "sushi"          : list((images_dir / "sushi").glob("*.jpg"))[0],
    "hamburger"      : list((images_dir / "hamburger").glob("*.jpg"))[0],
    "grilled_salmon" : list((images_dir / "grilled_salmon").glob("*.jpg"))[0],
}

print("🧪 Test Clarifai Food Model :\n")
for food_name, img_path in test_foods.items():
    results = clarifai_detect_food(str(img_path))
    if results:
        top3 = [f"{r['name']} ({r['score']}%)" for r in results[:3]]
        print(f"🍽️  {food_name:20s} → {' | '.join(top3)}")

🧪 Test Clarifai Food Model :

❌ Erreur : Account limits exceeded
❌ Erreur : Account limits exceeded
❌ Erreur : Account limits exceeded
❌ Erreur : Account limits exceeded
